#  Código usando Dask-Awkward

In [1]:
import uproot
import awkward as ak
import dask_awkward as dak
import dask
import numpy as np
from dask.diagnostics import ProgressBar

# ==============================================================================
# FUNÇÃO DE TRUTH MATCHING (RETORNA APENAS A MÁSCARA BOOLEANA)
# ==============================================================================
def get_signal_mask(pdg, mother):
    local_idx = ak.local_index(pdg)

    safe_mom = ak.where(mother >= 0, mother, 0)
    mom_pdg  = pdg[safe_mom]
    mom_pdg  = ak.where(mother >= 0, mom_pdg, -999)

    safe_grandmom = safe_mom[safe_mom]
    grandmom_pdg  = pdg[safe_grandmom]
    grandmom_pdg  = ak.where(safe_mom >= 0, grandmom_pdg, -999)

    B_ancestor_idx = ak.full_like(pdg, -1)
    B_ancestor_idx = ak.where((pdg == 511), local_idx, B_ancestor_idx)
    B_ancestor_idx = ak.where((pdg == 313) & (mom_pdg == 511), safe_mom, B_ancestor_idx)
    B_ancestor_idx = ak.where((pdg == 22)  & (mom_pdg == 511), safe_mom, B_ancestor_idx)

    is_K    = (pdg == 321)  & (mom_pdg == 313) & (grandmom_pdg == 511)
    is_pi   = (pdg == -211) & (mom_pdg == 313) & (grandmom_pdg == 511)
    is_mu_m = (pdg == 13)   & (mom_pdg == 22)  & (grandmom_pdg == 511)
    is_mu_p = (pdg == -13)  & (mom_pdg == 22)  & (grandmom_pdg == 511)

    B_ancestor_idx = ak.where(is_K,               safe_grandmom, B_ancestor_idx)
    B_ancestor_idx = ak.where(is_pi,              safe_grandmom, B_ancestor_idx)
    B_ancestor_idx = ak.where(is_mu_m | is_mu_p,  safe_grandmom, B_ancestor_idx)

    anc_kstar = B_ancestor_idx[(pdg == 313) & (mom_pdg == 511)]
    anc_K     = B_ancestor_idx[is_K]
    anc_pi    = B_ancestor_idx[is_pi]
    anc_mu_m  = B_ancestor_idx[is_mu_m]
    anc_mu_p  = B_ancestor_idx[is_mu_p]

    p_anc = B_ancestor_idx[:, :, np.newaxis]

    has_1_kstar = ak.sum(p_anc == anc_kstar[:, np.newaxis, :], axis=-1) == 1
    has_1_K     = ak.sum(p_anc == anc_K[:, np.newaxis, :],     axis=-1) == 1
    has_1_pi    = ak.sum(p_anc == anc_pi[:, np.newaxis, :],    axis=-1) == 1
    has_1_mu_m  = ak.sum(p_anc == anc_mu_m[:, np.newaxis, :],  axis=-1) == 1
    has_1_mu_p  = ak.sum(p_anc == anc_mu_p[:, np.newaxis, :],  axis=-1) == 1

    is_valid_chain  = has_1_kstar & has_1_K & has_1_pi & has_1_mu_m & has_1_mu_p
    particle_mask   = is_valid_chain & (B_ancestor_idx >= 0)

    # Retorna APENAS o jagged array de booleanos
    return particle_mask


# ==============================================================================
# SCRIPT PRINCIPAL
# ==============================================================================
file_path = "/cms/store/user/tdeandra/BPH_NanoAOD_MC/BdtoKstar2Mu_KstartoKPi_TuneCP5_13p6TeV_pythia8-evtgen/BPH_Nano_MC_BdtoKstarMuMu_NoFilter_2022/260427_132612/0000/BPH_MC_NANO_1.root"

print("Abrindo arquivo e montando grafo lazy...")

# API correta para carregar com dask_awkward
arrays = uproot.dask(
    {file_path: "Events"},
    expressions=["BPHGenPart_pdgId", "BPHGenPart_genPartIdxMother", "BPHGenPart_mass"],
    library="ak"
)

raw_pdg    = arrays["BPHGenPart_pdgId"]
raw_mother = arrays["BPHGenPart_genPartIdxMother"]
raw_mass   = arrays["BPHGenPart_mass"]

print(f"Número de partições: {raw_pdg.npartitions}")

# 1. Cria o grafo lazy da máscara mapeando a função pelas partições
mask_lazy = dak.map_partitions(
    get_signal_mask,
    raw_pdg,
    raw_mother
)

# 2. Aplica a máscara diretamente aos arrays originais
masked_pdg  = raw_pdg[mask_lazy]
masked_mass = raw_mass[mask_lazy]

# 3. Descobre quais eventos não estão vazios (possuem pelo menos 1 partícula de sinal)
event_has_signal = dak.num(masked_pdg, axis=1) > 0

# 4. Filtra eventos vazios
final_pdg  = masked_pdg[event_has_signal]
final_mass = masked_mass[event_has_signal]

print("\nExecutando com barra de progresso...")
with ProgressBar():
    # Computa tudo de uma vez no final usando o dask principal
    pdg_clean, mass_clean = dask.compute(final_pdg, final_mass)

print(f"\nTotal de eventos de SINAL retidos: {len(pdg_clean)}")

for i in range(min(5, len(pdg_clean))):
    print(f"\nEvento {i}:")
    print(f"  PDG  : {pdg_clean[i].tolist()}")
    print(f"  Mass : {mass_clean[i].tolist()}")

Abrindo arquivo e montando grafo lazy...
Número de partições: 1

Executando com barra de progresso...
[########################################] | 100% Completed | 102.72 ms
[########################################] | 100% Completed | 6.81 ss

Total de eventos de SINAL retidos: 185802

Evento 0:
  PDG  : [511, 313, -13, 22, 13, 22, 22, 22, 321, -211]
  Mass : [5.28125, 0.90625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.494140625, 0.0]

Evento 1:
  PDG  : [511, 313, -13, 22, 13, 22, 321, -211]
  Mass : [5.28125, 0.892578125, 0.0, 0.0, 0.0, 0.0, 0.494140625, 0.0]

Evento 2:
  PDG  : [511, 313, -13, 22, 13, 22, 22, 321, -211]
  Mass : [5.28125, 0.896484375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.494140625, 0.0]

Evento 3:
  PDG  : [511, 313, -13, 22, 13, 22, 22, 22, 22, 321, -211]
  Mass : [5.28125, 0.951171875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.494140625, 0.0]

Evento 4:
  PDG  : [511, 313, -13, 22, 13, 22, 321, -211]
  Mass : [5.28125, 0.923828125, 0.0, 0.0, 0.0, 0.0, 0.494140625, 0.0]


# Código usando Loop Python + Numba

In [2]:
import uproot
import awkward as ak
import numba as nb
import numpy as np

# ==============================================================================
# O MOTOR DE BUSCA GENEALÓGICO (Estritamente Matéria: B0 -> K*0)
# ==============================================================================
@nb.njit
def build_genealogy_masks(pdgs_events, moms_events):
    n_events = len(pdgs_events)
    event_mask = np.zeros(n_events, dtype=np.bool_)
    particle_masks = nb.typed.List()
    
    for i in range(n_events):
        pdgs = pdgs_events[i]
        moms = moms_events[i]
        n_parts = len(pdgs)
        
        p_mask = np.zeros(n_parts, dtype=np.bool_)
        found_signal = False
        
        for j in range(n_parts):
            # EXIGE ESTRITAMENTE MATÉRIA (511)
            if pdgs[j] == 511:
                b0_idx = j
                
                # 1. Procurar Filhas Diretas do B0
                kstar_idx = -1
                photon_indices = nb.typed.List.empty_list(nb.int64) 
                
                for k in range(n_parts):
                    if moms[k] == b0_idx:
                        # EXIGE ESTRITAMENTE K*0 MATÉRIA (313)
                        if pdgs[k] == 313:
                            kstar_idx = k
                        elif pdgs[k] == 22:
                            photon_indices.append(k)
                            
                if kstar_idx == -1 or len(photon_indices) == 0:
                    continue
                    
                # 2. Procurar Múons que sejam filhas dos Fótons
                muon_indices = nb.typed.List.empty_list(nb.int64)
                has_mu_plus = False
                has_mu_minus = False
                
                for k in range(n_parts):
                    if abs(pdgs[k]) == 13: # Múons sempre em par (+13 e -13)
                        is_from_photon = False
                        for p_idx in photon_indices:
                            if moms[k] == p_idx:
                                is_from_photon = True
                                break
                        
                        if is_from_photon:
                            muon_indices.append(k)
                            if pdgs[k] == -13: has_mu_plus = True
                            if pdgs[k] == 13:  has_mu_minus = True
                            
                # 3. Procurar Kaon e Pion filhos do K*0
                kaon_idx = -1
                pion_idx = -1
                for k in range(n_parts):
                    if moms[k] == kstar_idx:
                        # EXIGE ESTRITAMENTE K+ (321) e Pi- (-211)
                        if pdgs[k] == 321: kaon_idx = k
                        if pdgs[k] == -211: pion_idx = k
                        
                # 4. VEREDITO: A árvore está perfeita?
                if has_mu_plus and has_mu_minus and kaon_idx != -1 and pion_idx != -1:
                    found_signal = True
                    
                    # Salva as partículas dessa família na máscara
                    p_mask[b0_idx] = True
                    p_mask[kstar_idx] = True
                    p_mask[kaon_idx] = True
                    p_mask[pion_idx] = True
                    for p in photon_indices: p_mask[p] = True
                    for m in muon_indices: p_mask[m] = True
                    
        event_mask[i] = found_signal
        particle_masks.append(p_mask)
        
    return event_mask, particle_masks

# ==============================================================================
# EXECUÇÃO DO CÓDIGO PYTHON
# ==============================================================================
file_path = "/cms/store/user/tdeandra/BPH_NanoAOD_MC/BdtoKstar2Mu_KstartoKPi_TuneCP5_13p6TeV_pythia8-evtgen/BPH_Nano_MC_BdtoKstarMuMu_NoFilter_2022/260427_132612/0000/BPH_MC_NANO_1.root"
fname = file_path

print(f"Abrindo arquivo: {fname}")
file = uproot.open(fname)
tree = file["Events"]

print("Carregando arrays...")
pdgs = tree["BPHGenPart_pdgId"].array()
moms = tree["BPHGenPart_genPartIdxMother"].array()

print(f"Total de eventos antes: {len(pdgs)}")

print("\nVerificando a genealogia (apenas 511)...")
event_mask, particle_masks_numba = build_genealogy_masks(pdgs, moms)

ak_particle_mask = ak.Array(list(particle_masks_numba))

pdgs_validos = pdgs[event_mask]
mask_valida = ak_particle_mask[event_mask]

pdgs_final = pdgs_validos[mask_valida]

print(f"Total de eventos com topologia exata de matéria (511): {len(pdgs_final)}")

print("\n" + "="*70)
print("EXEMPLO DOS PRIMEIROS 5 EVENTOS (Árvore 100% limpa, só matéria):")
print("="*70)

for i in range(min(5, len(pdgs_final))):
    print(f"Evento {i}: {pdgs_final[i].tolist()}")

Abrindo arquivo: /cms/store/user/tdeandra/BPH_NanoAOD_MC/BdtoKstar2Mu_KstartoKPi_TuneCP5_13p6TeV_pythia8-evtgen/BPH_Nano_MC_BdtoKstarMuMu_NoFilter_2022/260427_132612/0000/BPH_MC_NANO_1.root
Carregando arrays...
Total de eventos antes: 375609

Verificando a genealogia (apenas 511)...
Total de eventos com topologia exata de matéria (511): 185802

EXEMPLO DOS PRIMEIROS 5 EVENTOS (Árvore 100% limpa, só matéria):
Evento 0: [511, 313, -13, 22, 13, 22, 22, 22, 321, -211]
Evento 1: [511, 313, -13, 22, 13, 22, 321, -211]
Evento 2: [511, 313, -13, 22, 13, 22, 22, 321, -211]
Evento 3: [511, 313, -13, 22, 13, 22, 22, 22, 22, 321, -211]
Evento 4: [511, 313, -13, 22, 13, 22, 321, -211]
